# Compute HelpSteer2 Coefficients

This notebook computes coefficient vectors for the current HelpSteer2 experiment from the saved relationship matrix $R$.

It runs the all-method coefficient script for the direct-preference and uniform baselines plus:

- M1: MGDA-inspired one-shot mapping,
- M2: preference-weighted alpha-MGDA variant,
- C1: trust-region CAGrad-inspired mapping,
- C2: soft-min CAGrad-inspired mapping,
- P1: conflict-weighted closed-form shrinkage,
- P2: PCGrad-inspired reconstruction with deterministic strongest-conflict ordering.

The outputs are written under `results/` as small CSV and JSON files.

## 1. Clone or update the repository

This cell starts in `/content`, updates `/content/master-thesis` if it is already a Git repository, and otherwise clones the repository. It avoids nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## 2. Show repository structure

Confirm that the active folders and coefficient script are available.

In [ ]:
!pwd
!ls
!ls scripts
!ls results || echo "No results folder found yet."

## 3. Install lightweight dependencies

The coefficient script uses NumPy and SciPy for simplex-constrained optimization and pandas for table preview.

In [ ]:
!pip install -q "pandas==2.2.2" "numpy<2.1" scipy

## 4. Check the relationship matrix

The coefficient computation reads `results/helpsteer2_relationship_matrix.csv`. Notebook 11 creates this file from the five trained HelpSteer2 LoRA adapters.

In [ ]:
from pathlib import Path

matrix_path = Path("results/helpsteer2_relationship_matrix.csv")

if not matrix_path.is_file():
    raise FileNotFoundError(
        f"Missing {matrix_path}. Run Notebook 11 or "
        "scripts/compute_helpsteer2_relationship_matrix.py first."
    )

print(f"Relationship matrix found: {matrix_path}")

In [ ]:
import pandas as pd

relationship_df = pd.read_csv(matrix_path)
display(relationship_df)

## 5. Run coefficient computation

The script computes direct-preference, uniform, M1, M2, C1, C2, P1, and P2 coefficients for the active HelpSteer2 preference vectors. It validates that every lambda vector is non-negative, has the correct dimension, and sums to one.

In [ ]:
!python scripts/compute_helpsteer2_all_method_coefficients.py

## 6. Inspect outputs

The CSV contains one row per preference vector and method. The metadata JSON records method definitions, hyperparameters, and validation checks.

In [ ]:
!ls results | grep helpsteer2_all_method_coefficients

In [ ]:
coefficients_path = Path("results/helpsteer2_all_method_coefficients.csv")
coefficients_df = pd.read_csv(coefficients_path)
print(f"Rows: {len(coefficients_df)}")
display(coefficients_df)

In [ ]:
import json

metadata_path = Path("results/helpsteer2_all_method_coefficients_metadata.json")
with metadata_path.open("r", encoding="utf-8") as metadata_file:
    metadata = json.load(metadata_file)

print(json.dumps(metadata, indent=2)[:3000])

## 7. Git safety check

Small CSV and JSON result files may be committed when they document an experiment. Do not commit generated adapter folders, checkpoints, zip files, safetensors, binary model weights, or other large model files.

In [ ]:
!git status